# 06 - Build Patient Exam Sequences for LSTM

This notebook converts extracted CNN image features into patient-level exam sequences.

Each exam/session is organised into fixed mammogram view slots: L-CC, R-CC, L-MLO, and R-MLO. Missing views are handled using masks. The notebook also creates asymmetry features and recency weights so the LSTM can learn how patient breast imaging patterns change over time.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

In [4]:
# Set file paths

FEATURE_PATH = "/content/drive/MyDrive/EMBED/Features/embed_cnn_features.npy"
METADATA_PATH = "/content/drive/MyDrive/EMBED/embed_cnn_feature_metadata.csv"

SEQUENCE_OUTPUT_PATH = "/content/drive/MyDrive/EMBED/Features/embed_lstm_patient_sequences.npy"
LABEL_OUTPUT_PATH = "/content/drive/MyDrive/EMBED/Features/embed_lstm_patient_labels.npy"
PATIENT_METADATA_OUTPUT_PATH = "/content/drive/MyDrive/EMBED/Features/embed_lstm_patient_metadata.csv"

In [5]:
# Load extracted CNN features and metadata

cnn_features = np.load(FEATURE_PATH)
metadata = pd.read_csv(METADATA_PATH)

print("CNN feature shape:", cnn_features.shape)
print("Metadata shape:", metadata.shape)

metadata.head()

CNN feature shape: (1038, 2048)
Metadata shape: (1038, 11)


,empi_anon,acc_anon,study_date_anon,ViewPosition,ImageLateralityFinal,processed_image_path,risk_1yr,risk_2yr,risk_3yr,risk_4yr,risk_5yr
0,57769289,3512912135438605,2016-06-15 00:00:00,CC,R,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1
1,57769289,3504436559522696,2017-01-21 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1
2,57769289,3504436559522696,2017-01-21 00:00:00,CC,L,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1
3,57769289,3512912135438605,2016-06-15 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1
4,57769289,6528636244388176,2016-07-20 00:00:00,CC,L,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1


In [6]:
metadata = metadata.copy()

metadata["feature_idx"] = np.arange(len(metadata))

print(metadata.shape)

metadata.head()

(1038, 12)


,empi_anon,acc_anon,study_date_anon,ViewPosition,ImageLateralityFinal,processed_image_path,risk_1yr,risk_2yr,risk_3yr,risk_4yr,risk_5yr,feature_idx
0,57769289,3512912135438605,2016-06-15 00:00:00,CC,R,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1,0
1,57769289,3504436559522696,2017-01-21 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1,1
2,57769289,3504436559522696,2017-01-21 00:00:00,CC,L,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1,2
3,57769289,3512912135438605,2016-06-15 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1,3
4,57769289,6528636244388176,2016-07-20 00:00:00,CC,L,/content/drive/MyDrive/EMBED/processed_images_...,0,1,1,1,1,4


In [7]:
# Create view slot names

metadata["view_slot"] = (
    metadata["ImageLateralityFinal"]
    + "_"
    + metadata["ViewPosition"]
)

print(
    metadata["view_slot"]
    .value_counts()
)

view_slot
R_MLO    272
L_CC     259
L_MLO    256
R_CC     251
Name: count, dtype: int64


In [8]:
print(
    sorted(
        metadata["view_slot"]
        .unique()
    )
)

['L_CC', 'L_MLO', 'R_CC', 'R_MLO']


In [9]:
# Convert dates

metadata["study_date_anon"] = pd.to_datetime(
    metadata["study_date_anon"]
)

print(
    metadata["study_date_anon"].min()
)

print(
    metadata["study_date_anon"].max()
)

2012-12-03 00:00:00
2020-11-06 00:00:00


In [10]:
# Group images into exams

exam_groups = metadata.groupby(
    ["empi_anon", "study_date_anon"]
)

print("Total exams:", len(exam_groups))

Total exams: 229


In [11]:
# Look at one exam

sample_key, sample_exam = next(iter(exam_groups))

print("Patient:", sample_key[0])
print("Exam Date:", sample_key[1])

sample_exam[
    [
        "view_slot",
        "feature_idx"
    ]
]

Patient: 11057159
Exam Date: 2013-03-05 00:00:00


,view_slot,feature_idx
994,R_CC,994
995,L_MLO,995
996,R_MLO,996
997,R_CC,997
999,L_MLO,999
1008,L_CC,1008
1011,L_CC,1011
1015,R_MLO,1015


In [12]:
# Number of views per exam

exam_view_counts = (
    metadata
    .groupby(
        ["empi_anon", "study_date_anon"]
    )["view_slot"]
    .nunique()
)

print(exam_view_counts.describe())

print("\nView count distribution:")

print(
    exam_view_counts
    .value_counts()
    .sort_index()
)

count    229.000000
mean       3.292576
std        1.028961
min        1.000000
25%        2.000000
50%        4.000000
75%        4.000000
max        4.000000
Name: view_slot, dtype: float64

View count distribution:
view_slot
1     11
2     64
3      1
4    153
Name: count, dtype: int64


In [13]:
# How many images per view slot within an exam?

view_slot_counts = (
    metadata
    .groupby(
        ["empi_anon",
         "study_date_anon",
         "view_slot"]
    )
    .size()
)

print(view_slot_counts.describe())

print("\nDistribution:")

print(
    view_slot_counts
    .value_counts()
    .sort_index()
)

count    754.000000
mean       1.376658
std        0.644755
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        5.000000
dtype: float64

Distribution:
1    528
2    177
3     41
4      7
5      1
Name: count, dtype: int64


In [14]:
# Helper function

def get_view_feature(exam_df, view_slot):

    rows = exam_df[
        exam_df["view_slot"] == view_slot
    ]

    if len(rows) == 0:
        return None

    feature_indices = rows["feature_idx"].values

    features = cnn_features[
        feature_indices
    ]

    return features.mean(axis=0)

In [15]:
sample_key, sample_exam = next(iter(exam_groups))

l_cc = get_view_feature(
    sample_exam,
    "L_CC"
)

r_cc = get_view_feature(
    sample_exam,
    "R_CC"
)

l_mlo = get_view_feature(
    sample_exam,
    "L_MLO"
)

r_mlo = get_view_feature(
    sample_exam,
    "R_MLO"
)

print("L_CC:", None if l_cc is None else l_cc.shape)
print("R_CC:", None if r_cc is None else r_cc.shape)
print("L_MLO:", None if l_mlo is None else l_mlo.shape)
print("R_MLO:", None if r_mlo is None else r_mlo.shape)

L_CC: (2048,)
R_CC: (2048,)
L_MLO: (2048,)
R_MLO: (2048,)


In [16]:
# Build one exam-level feature

def build_exam_feature(exam_df, recency_weight=1.0):
    view_slots = ["L_CC", "R_CC", "L_MLO", "R_MLO"]

    view_features = {}
    view_mask = []

    for slot in view_slots:
        feature = get_view_feature(exam_df, slot)
        view_features[slot] = feature
        view_mask.append(0 if feature is None else 1)

    available_features = [
        feature for feature in view_features.values()
        if feature is not None
    ]

    exam_feature = np.mean(available_features, axis=0)

    zero_feature = np.zeros(2048)

    if view_features["L_CC"] is not None and view_features["R_CC"] is not None:
        cc_asymmetry = np.abs(view_features["L_CC"] - view_features["R_CC"])
    else:
        cc_asymmetry = zero_feature

    if view_features["L_MLO"] is not None and view_features["R_MLO"] is not None:
        mlo_asymmetry = np.abs(view_features["L_MLO"] - view_features["R_MLO"])
    else:
        mlo_asymmetry = zero_feature

    final_exam_feature = np.concatenate([
        exam_feature,
        cc_asymmetry,
        mlo_asymmetry,
        np.array(view_mask),
        np.array([recency_weight])
    ])

    return final_exam_feature

In [17]:
sample_feature = build_exam_feature(sample_exam)

print(sample_feature.shape)

(6149,)


In [18]:
# Calculate recency weights for each exam

def calculate_recency_weights(patient_exam_dates):
    latest_date = max(patient_exam_dates)

    weights = {}

    for exam_date in patient_exam_dates:
        days_before_latest = (latest_date - exam_date).days
        years_before_latest = days_before_latest / 365.25

        raw_weight = np.exp(-0.5 * years_before_latest)

        weights[exam_date] = raw_weight

    total_weight = sum(weights.values())

    normalized_weights = {
        date: weight / total_weight
        for date, weight in weights.items()
    }

    return normalized_weights

In [19]:
# Build patient-level sequences

patient_sequences = []
patient_labels = []
patient_metadata_rows = []

for patient_id, patient_df in metadata.groupby("empi_anon"):

    patient_dates = sorted(patient_df["study_date_anon"].unique())

    recency_weights = calculate_recency_weights(patient_dates)

    exam_features = []

    for exam_date in patient_dates:
        exam_df = patient_df[
            patient_df["study_date_anon"] == exam_date
        ]

        exam_feature = build_exam_feature(
            exam_df,
            recency_weight=recency_weights[exam_date]
        )

        exam_features.append(exam_feature)

    patient_sequence = np.stack(exam_features)

    label_values = patient_df[
        ["risk_1yr", "risk_2yr", "risk_3yr", "risk_4yr", "risk_5yr"]
    ].iloc[0].values.astype(np.float32)

    patient_sequences.append(patient_sequence)
    patient_labels.append(label_values)

    patient_metadata_rows.append({
        "empi_anon": patient_id,
        "num_exams": len(patient_dates),
        "first_exam_date": min(patient_dates),
        "last_exam_date": max(patient_dates)
    })

print("Patients:", len(patient_sequences))
print("Example sequence shape:", patient_sequences[0].shape)
print("Example label:", patient_labels[0])

Patients: 40
Example sequence shape: (7, 6149)
Example label: [0. 0. 0. 0. 0.]


In [20]:
# Pad patient sequences to the same length

max_sequence_length = max(
    sequence.shape[0]
    for sequence in patient_sequences
)

feature_size = patient_sequences[0].shape[1]

padded_sequences = np.zeros(
    (
        len(patient_sequences),
        max_sequence_length,
        feature_size
    ),
    dtype=np.float32
)

sequence_masks = np.zeros(
    (
        len(patient_sequences),
        max_sequence_length
    ),
    dtype=np.float32
)

for i, sequence in enumerate(patient_sequences):
    seq_len = sequence.shape[0]

    padded_sequences[i, :seq_len, :] = sequence
    sequence_masks[i, :seq_len] = 1

patient_labels = np.array(
    patient_labels,
    dtype=np.float32
)

print("Padded sequences shape:", padded_sequences.shape)
print("Sequence masks shape:", sequence_masks.shape)
print("Labels shape:", patient_labels.shape)
print("Max sequence length:", max_sequence_length)

Padded sequences shape: (40, 13, 6149)
Sequence masks shape: (40, 13)
Labels shape: (40, 5)
Max sequence length: 13


In [21]:
# Save LSTM-ready sequence data

np.save(SEQUENCE_OUTPUT_PATH, padded_sequences)
np.save(LABEL_OUTPUT_PATH, patient_labels)

patient_metadata = pd.DataFrame(patient_metadata_rows)
patient_metadata["max_sequence_length"] = max_sequence_length

patient_metadata.to_csv(
    PATIENT_METADATA_OUTPUT_PATH,
    index=False
)

print("Saved sequences:", SEQUENCE_OUTPUT_PATH)
print("Saved labels:", LABEL_OUTPUT_PATH)
print("Saved metadata:", PATIENT_METADATA_OUTPUT_PATH)

Saved sequences: /content/drive/MyDrive/EMBED/Features/embed_lstm_patient_sequences.npy
Saved labels: /content/drive/MyDrive/EMBED/Features/embed_lstm_patient_labels.npy
Saved metadata: /content/drive/MyDrive/EMBED/Features/embed_lstm_patient_metadata.csv


In [22]:
MASK_OUTPUT_PATH = "/content/drive/MyDrive/EMBED/Features/embed_lstm_sequence_masks.npy"

np.save(MASK_OUTPUT_PATH, sequence_masks)

print("Saved masks:", MASK_OUTPUT_PATH)

Saved masks: /content/drive/MyDrive/EMBED/Features/embed_lstm_sequence_masks.npy
